In [4]:
import os
# Simple CPU16 Assembler
# Supports:
# LOAD, ADD, SUB, MUL, OUT, NOP
# AND a custom MATMUL instruction for matrix multiplication
#
# Example assembly input:
#
# LOAD 15
# ADD 15
# SUB 3
# MUL 11
# OUT
# NOP
#
# MATMUL format:
# MATMUL A00 A01 A10 A11 B00 B01 B10 B11
#
# Example:
# MATMUL 1 2 3 4 5 6 7 8
#
# Output is written as hexadecimal machine code lines
# suitable for Vivado $readmemh()

OPCODES = {
    "NOP":    0x0,
    "LOAD":   0x1,
    "ADD":    0x2,
    "OUT":    0x3,
    "SUB":    0x4,
    "MUL":    0x5,
    "MATMUL": 0x6,   # custom opcode
}


def encode_simple(op, imm=0):
    """
    Encode normal immediate instruction:
    opcode in [15:12]
    immediate in [3:0]

    Example:
    LOAD 15 -> 100F
    """
    opcode = OPCODES[op]
    imm = int(imm) & 0xF
    instr = (opcode << 12) | imm
    return f"{instr:04X}"



def encode_matmul(values):
    """
    MATMUL uses multiple words.

    First word:
        6000 = start MATMUL instruction

    Then store 8 matrix values:
        A00 A01 A10 A11
        B00 B01 B10 B11

    Each value is emitted as one 16-bit word.

    This is much easier for hardware decode.
    """
    if len(values) != 8:
        raise ValueError(
            "MATMUL requires exactly 8 values: "
            "A00 A01 A10 A11 B00 B01 B10 B11"
        )

    output = ["6000"]

    for v in values:
        value = int(v) & 0xFFFF
        output.append(f"{value:04X}")

    return output



def assemble_line(line):
    """
    Assemble one line of assembly.
    Returns list of hex words.
    """
    line = line.strip()

    if not line:
        return []

    if line.startswith("#"):
        return []

    parts = line.replace(",", " ").split()
    op = parts[0].upper()

    if op not in OPCODES:
        raise ValueError(f"Unknown instruction: {op}")

    if op in ["NOP", "OUT"]:
        return [encode_simple(op, 0)]

    elif op in ["LOAD", "ADD", "SUB", "MUL"]:
        if len(parts) != 2:
            raise ValueError(f"{op} requires 1 immediate value")
        return [encode_simple(op, parts[1])]

    elif op == "MATMUL":
        return encode_matmul(parts[1:])

    else:
        raise ValueError(f"Unsupported instruction: {op}")



def assemble_file(input_file, output_file):
    machine_code = []

    with open(input_file, "r") as f:
        for line_number, line in enumerate(f, start=1):
            try:
                encoded = assemble_line(line)
                machine_code.extend(encoded)
            except Exception as e:
                print(f"Line {line_number}: {e}")
                return

    with open(output_file, "w") as f:
        for word in machine_code:
            f.write(word + "\n")

    print("Assembly successful")
    print(f"Output written to: {output_file}")


if __name__ == "__main__":
    print("CPU16 Assembler")
    print("----------------")

    input_file = "E:/vhdl programs/CPU16modular/CPU16modular.srcs/sim_1/new/input.txt"
    output_file = "E:/vhdl programs/CPU16modular/CPU16modular.srcs/sim_1/new/program.txt"

    assemble_file(input_file, output_file)

CPU16 Assembler
----------------
Assembly successful
Output written to: E:/vhdl programs/CPU16modular/CPU16modular.srcs/sim_1/new/program.txt
